# 05 - Master notebook: the whole workflow from the .py modules

Rubric: *Automation*. This notebook only calls functions from `setalpha.pipeline`, the same code the CLI
(`scripts/run_pipeline.py`) and the Airflow DAGs run.

* `MODE = "file"` uses the local parquet store (no network); `MODE = "live"` downloads only the new days.
* `RETRAIN = False` reuses the saved models / tuned parameters; set it to True to run the walk-forward again.

In [ ]:
%load_ext autoreload
%autoreload 2
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
from setalpha import config, ingest, transform, model, regime, portfolio, strategy, evaluate, report, pipeline
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 180)
pd.set_option("display.float_format", "{:.4f}".format)
plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True})
import logging; logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s", force=True)
MODE, RETRAIN = "file", False

In [ ]:
raw = pipeline.step_ingest(MODE)                 # 1. download / load raw data
df = pipeline.step_transform(raw)                # 2. factor model, features, targets -> one dataset
if RETRAIN:
    pipeline.step_train(retune=False)            # 3. walk-forward models + production refit
    pipeline.step_simulate()                     # 4. vector + exact simulations

In [ ]:
signals = pipeline.step_signals()                # 5. next-day trades from the production models
signals[signals["target_weight_%"] > 0]

In [ ]:
path = pipeline.step_report()                    # 6. HTML dashboard
print("dashboard:", path)
joblib.load(pipeline.SIM_RESULTS)["metrics"][["CAGR %", "Sharpe", "max DD %", "alpha % p.a.", "DSR"]].round(3)